In [ ]:
"""Cross-group multi-model experiment with a paired authority manipulation."""
from __future__ import annotations

import argparse
import csv
import json
import random
import sys
from pathlib import Path
from typing import Any

from langchain_community.chat_models import ChatOllama
from persona_v2 import generate_persona, upgrade_to_authority

TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議"
AGENTS = 6
ROUNDS = 5
REPETITIONS = 25
INITIAL_STANCES = ["Support"] * 4 + ["Oppose"] * 2
STANCE_VALUES = {-2, -1, 0, 1, 2}
ENABLE_MEMORY = False  # True: 記憶/反思實驗因素；False: 維持原始流程


def make_prompt(persona: Any, history: list[dict[str, Any]], authority: bool, score: int, memory_context: str = "") -> str:
    public_history = "\n".join(
        f"Round {item['round']} | {item['agent']}: {item['public_post']}"
        for item in history
    ) or "（這是第一回合，尚無前回合留言。）"
    authority_note = (
        "本場有一名被標記的領域權威；你可以評估其論點，但仍須根據證據自行判斷。"
        if authority else "本場沒有權威角色，所有代理人的社會地位相同。"
    )
    if memory_context:
        public_history += f"\n\n代理人可用的長期記憶摘要：\n{memory_context}"
    return f"""你正在參與 BBS 社會議題討論。
議題：{TOPIC}
你的角色：{persona.name}
目前立場分數：{score}（-2 強烈反對，-1 反對，0 中立，+1 支持，+2 強烈支持）
{authority_note}
請只輸出 JSON，不要輸出 Markdown：
{{"stance_score": -2, "rationale": "簡短的立場判斷摘要", "public_post": "給其他代理人看的公開留言"}}
公開留言應自然回應前一回合內容，但不要聲稱自己讀到不存在的資訊。
前一回合完整留言：
{public_history}
"""


def invoke_agent(llm: ChatOllama, persona: Any, history: list[dict[str, Any]], authority: bool, score: int, memory_context: str = "") -> dict[str, Any]:
    raw = str(llm.invoke(make_prompt(persona, history,
              authority, score, memory_context)).content).strip()
    try:
        payload = json.loads(raw[raw.find("{"): raw.rfind("}") + 1])
        new_score = int(payload["stance_score"])
        if new_score not in STANCE_VALUES:
            raise ValueError("stance_score must be between -2 and 2")
        return {"stance_score": new_score, "rationale": str(payload.get("rationale", "")), "public_post": str(payload["public_post"])}
    except Exception:
        return {"stance_score": score, "rationale": "parser fallback", "public_post": raw}


def build_memory_context(memory_stream: Any) -> str:
    relevant = memory_stream.search_by_vector(TOPIC, k=3)
    reflections = [
        m for m in memory_stream.memories if m.content.startswith("[Reflection]")][-2:]
    return "\n".join(f"- {m.content}" for m in relevant + reflections)


def polarization(scores: list[int]) -> float:
    return sum(abs(s) for s in scores) / len(scores)


def fragmentation(scores: list[int]) -> float:
    support = sum(s > 0 for s in scores)
    oppose = sum(s < 0 for s in scores)
    return 0.0 if support + oppose == 0 else 1 - abs(support - oppose) / (support + oppose)


def metrics(records: list[dict[str, Any]]) -> dict[str, Any]:
    by_round = [[int(x["stance_score"]) for x in records if x["round"] == r]
                for r in range(1, ROUNDS + 1)]
    p = [polarization(scores) for scores in by_round]
    f = [fragmentation(scores) for scores in by_round]
    aligned = 0
    for r in range(1, ROUNDS):
        current = by_round[r]
        majority = 1 if sum(current) > 0 else -1 if sum(current) < 0 else 0
        if majority == 0:
            continue
        previous = {x["agent"]: int(x["stance_score"])
                    for x in records if x["round"] == r}
        for item in (x for x in records if x["round"] == r + 1):
            old = previous[item["agent"]]
            new = int(item["stance_score"])
            if abs(new - 2 * majority) < abs(old - 2 * majority):
                aligned += 1
    return {"CR": aligned / (AGENTS * (ROUNDS - 1)), "P": p, "delta_P": p[-1] - p[0], "F": f, "F5": f[-1]}


def write_outputs(output_dir: Path, rows: list[dict[str, Any]], dialogues: list[dict[str, Any]]) -> None:
    output_dir.mkdir(exist_ok=True)
    fields = ["run", "condition", "seed", "authority", "memory_enabled",
              "model_assignment", "CR", "delta_P", "F5", "P", "F"]
    with (output_dir / "metrics.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    with (output_dir / "dialogues.jsonl").open("w", encoding="utf-8") as f:
        for row in dialogues:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


MODEL_POOL = ["llama3:8b", "qwen3:8b",
              "deepseek-r1:8b", "qwen2.5:32b-instruct"]
OUTPUT_DIR = Path("experiment-cross-authority-result")


def run_condition(run_id: int, condition: str, seed: int):
    rng = random.Random(seed)
    stances = INITIAL_STANCES[:]
    rng.shuffle(stances)
    personas = {
        f"Agent_{i + 1}": generate_persona(f"Agent_{i + 1}", stances[i]) for i in range(AGENTS)}
    authority_slot = rng.randrange(AGENTS)
    authority_name = list(personas)[
        authority_slot] if condition == "authority" else None
    if authority_name:
        personas[authority_name] = upgrade_to_authority(
            personas[authority_name], TOPIC)
    models = MODEL_POOL[:] + [rng.choice(MODEL_POOL), rng.choice(MODEL_POOL)]
    rng.shuffle(models)
    agents = {name: ChatOllama(
        model=models[i], temperature=0.7) for i, name in enumerate(personas)}
    memory_streams = {}
    reflection_engines = {}
    if ENABLE_MEMORY:
        from memory import MemoryStream
        from reflection import ReflectionEngine
        memory_streams = {name: MemoryStream() for name in personas}
        reflection_engines = {name: ReflectionEngine(
            agents[name], memory_streams[name], reflection_threshold=20) for name in personas}
    assigned = dict(zip(personas, models))
    current = {name: (1 if p.initial_stance == "Support" else -1)
               for name, p in personas.items()}
    records = []
    previous_round = []
    for round_id in range(1, ROUNDS + 1):
        current_round = []
        for name, persona in personas.items():
            memory_context = build_memory_context(
                memory_streams[name]) if ENABLE_MEMORY else ""
            result = invoke_agent(agents[name], persona, previous_round,
                                  condition == "authority", current[name], memory_context)
            current[name] = result["stance_score"]
            if ENABLE_MEMORY:
                for observer_name, stream in memory_streams.items():
                    importance = 3 if name == authority_name else 1
                    stream.add_memory(
                        f"Round {round_id} | {name}: {result['public_post']}", importance=importance)
                reflection_engines[name].reflect()
            current_round.append({"run": run_id, "condition": condition, "round": round_id, "agent": name,
                                 "model": assigned[name], "is_authority": name == authority_name, "memory_enabled": ENABLE_MEMORY, **result})
        records.extend(current_round)
        previous_round = current_round
    return records, {"run": run_id, "condition": condition, "seed": seed, "authority": authority_name or "", "memory_enabled": ENABLE_MEMORY, "model_assignment": json.dumps(assigned, ensure_ascii=False), **metrics(records)}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--condition", choices=("control",
                        "authority", "both"), default="both")
    parser.add_argument("--repetitions", type=int, default=REPETITIONS)
    args = parser.parse_args(
        args=[] if "ipykernel" in sys.modules else None)
    OUTPUT_DIR.mkdir(exist_ok=True)
    rows = []
    dialogues = []
    conditions = ("control", "authority") if args.condition == "both" else (
        args.condition,)
    for run_id in range(1, args.repetitions + 1):
        for condition in conditions:
            records, summary = run_condition(run_id, condition, 3000 + run_id)
            rows.append(summary)
            dialogues.extend(records)
    fields = ["run", "condition", "seed", "authority", "memory_enabled",
              "model_assignment", "CR", "delta_P", "F5", "P", "F"]
    with (OUTPUT_DIR / "metrics.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    with (OUTPUT_DIR / "dialogues.jsonl").open("w", encoding="utf-8") as f:
        for row in dialogues:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"saved {len(rows)} condition-runs to {OUTPUT_DIR}")


if __name__ == "__main__":
    main()